# ELO Scores

## Objective

Add the 2026 starting ELO score to each 2026 World Cup team. I use the distance dataset created in this folder and join only the `WC_2026` ELO score from `ELOStartRatings.csv`.

## Inputs

- `1.DataCleaning-R/Data/RDS/WC2026DistanceFromHost.rds`
- `1.DataCleaning-R/Data/CSV/ELOStartRatings.csv`

## Output

- `1.DataCleaning-R/Data/RDS/WC2026DistanceELO.rds`

## Acknowledgments

ELO data is taken from eloratings.net.

## Libraries

In [38]:
library(here)
library(tidyverse)

## Load Data

The distance dataset already contains one row for each 2026 World Cup team.

In [39]:
distance_from_host <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "WC2026DistanceFromHost.rds"))

elo_scores <- read.csv(here("1.DataCleaning-R", "Data", "CSV", "ELOStartRatings.csv")) %>%
    select(elo_team = team, elo_2026 = WC_2026)

distance_from_host %>% count(tournament_id)
elo_scores %>% slice_sample(n = 5)

tournament_id,n
<chr>,<int>
WC-2026,48


elo_team,elo_2026
<chr>,<int>
Denmark,1864
Sweden,1660
Colombia,1998
Italy,1859
Scotland,1790


## Match Team Names

A few country names need to match the names used in the ELO file.

In [40]:
elo_name <- function(country) {
    case_when(
        country == "Cabo Verde" ~ "Cape Verde",
        country == "Congo DR" ~ "DR Congo",
        TRUE ~ country
    )
}

distance_elo_names <- distance_from_host %>%
    mutate(elo_team = elo_name(team_name))

distance_elo_names %>%
    anti_join(elo_scores, by = "elo_team") %>%
    select(team_name, team_code, elo_team)

team_name,team_code,elo_team
<chr>,<chr>,<chr>


## Add ELOs

Only the 2026 starting ELO score is added. Attack and defence ELOs are not included.

In [41]:
distance_elo <- distance_elo_names %>%
    left_join(elo_scores, by = "elo_team") %>%
    select(
        tournament_id,
        team_name,
        team_id,
        team_code,
        host_country,
        is_home_country,
        country_lat,
        country_long,
        host_lat,
        host_long,
        distance_from_host_km,
        elo_2026
    ) %>%
    arrange(team_name)

distance_elo %>%
    summarize(
        teams = n(),
        missing_elo = sum(is.na(elo_2026)),
        min_elo = min(elo_2026, na.rm = TRUE),
        max_elo = max(elo_2026, na.rm = TRUE),
        .groups = "drop"
    )

teams,missing_elo,min_elo,max_elo
<int>,<int>,<int>,<int>
48,0,1425,2172


In [42]:
distance_elo <- distance_elo %>%
    select(team_name, team_id, team_code, distance_from_host_km, elo_2026)

distance_elo


team_name,team_id,team_code,distance_from_host_km,elo_2026
<chr>,<chr>,<chr>,<dbl>,<int>
Algeria,T-01,DZA,8467.886,1757
Argentina,T-03,ARG,9076.351,2113
Australia,T-04,AUS,14653.989,1774
Austria,T-05,AUT,8351.637,1818
Belgium,T-06,BEL,7608.706,1850
Bosnia and Herzegovina,T-08,BIH,8799.915,1572
Brazil,T-09,BRA,7396.652,1978
Cabo Verde,NA,CPV,7732.128,1561
Canada,T-12,CAN,0.000,1802


Manually compare to website

In [44]:
set.seed(2026)
distance_elo %>%
    slice_sample(n=5)

team_name,team_id,team_code,distance_from_host_km,elo_2026
<chr>,<chr>,<chr>,<dbl>,<int>
Netherlands,T-48,NLD,7554.561,1959
Paraguay,T-55,PRY,8273.165,1833
Senegal,T-65,SEN,8646.943,1807
Turkey,T-80,TUR,10136.437,1880
Mexico,T-46,MEX,0.000,1835


All good!

## Save

In [43]:
saveRDS(distance_elo, here("1.DataCleaning-R", "Data", "RDS", "WC2026DistanceELO.rds"))